# Module 02 — Lecture 3: The Optimization Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_02_memory_optimization/03_optimization_workflow.ipynb)

---

Optimization without measurement is guesswork. This lecture gives you the systematic workflow to identify and fix performance bottlenecks in any CUDA kernel.

**Learning objectives:**
- Use `nvprof` and Nsight Compute to profile kernels
- Interpret the roofline model to classify kernels as compute-bound or memory-bound
- Apply occupancy analysis to choose block sizes
- Combine all Module 02 techniques into an optimization checklist

In [ ]:
!nvidia-smi

## 1. The Roofline Model

The roofline model tells you the theoretical maximum performance of a kernel given its arithmetic intensity.

```
Performance                  ↑
(GFLOPS)                     │         ← Compute ceiling (e.g., 8000 GFLOPS)
                             │  ─────────────────────────────────────────
                             │  /
                             │ /  ← Memory bandwidth slope
                             │/      GFLOPS = AI × BW (GB/s)
                             ───────────────────────────────────────────▶
                          Arithmetic Intensity (FLOPs/byte)

Ridge point: AI = Compute / BW
  e.g., T4: 8100 GFLOPS / 300 GB/s = 27 FLOPs/byte

If AI < 27: kernel is MEMORY-BOUND → optimize memory access patterns
If AI > 27: kernel is COMPUTE-BOUND → optimize math / use tensor cores
```

### Computing Arithmetic Intensity for Common Operations

| Operation | FLOPs | Bytes | AI |
|-----------|-------|-------|----|
| Vector add C=A+B | 1 FLOP/elem | 12 bytes/elem | 0.08 → memory-bound |
| LIF neuron update | ~8 FLOPs/elem | 8 bytes/elem (V, I) | 1.0 → memory-bound |
| Dense matmul N×N | 2N FLOPs/elem | (12/N) bytes/elem | N/6 → compute-bound for N>150 |
| H-H gating ODEs | ~50 FLOPs/elem | 32 bytes/elem | 1.6 → memory-bound |

**Key insight:** Most neuroscience kernels (LIF, H-H, spike detection) are **memory-bound**. The optimization focus should be on coalescing and caching, not on math.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Roofline model for NVIDIA T4
BW_peak   = 300.0    # GB/s (T4 memory bandwidth)
FP32_peak = 8100.0   # GFLOPS (T4 FP32 peak)
ridge = FP32_peak / BW_peak  # ~27 FLOPs/byte

AI = np.logspace(-2, 3, 500)
perf = np.minimum(AI * BW_peak, FP32_peak)

# Our kernels
kernels = {
    'Vector add': (0.08, 30),
    'LIF update': (1.0, 280),
    'HH gating': (1.6, 480),
    'Dense matmul\n(N=512)': (85.3, 5200),
    'Sparse matvec\n(10% density)': (0.5, 150),
}

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(AI, perf, 'k-', linewidth=2.5, label='Roofline (T4)')
ax.axvline(ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge point ({ridge:.0f} FLOPs/byte)')

colors = plt.cm.Set1(np.linspace(0, 0.8, len(kernels)))
for (name, (ai, perf_val)), color in zip(kernels.items(), colors):
    ax.scatter([ai], [perf_val], s=120, color=color, zorder=5)
    ax.annotate(name, (ai, perf_val), textcoords='offset points',
                xytext=(8, 0), fontsize=9, color=color)

ax.fill_between(AI[AI < ridge], perf[AI < ridge], alpha=0.05, color='blue',
                label='Memory-bound region')
ax.fill_between(AI[AI >= ridge], perf[AI >= ridge], alpha=0.05, color='red',
                label='Compute-bound region')

ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=13)
ax.set_ylabel('Performance (GFLOPS)', fontsize=13)
ax.set_title('Roofline Model — NVIDIA T4 GPU\nNeuroscience Kernel Positions', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.01, 1000)
ax.set_ylim(1, 15000)

plt.tight_layout()
plt.savefig('roofline_neuroscience.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"T4 ridge point: {ridge:.1f} FLOPs/byte")
print("Takeaway: all neuron-update kernels are memory-bound → focus on bandwidth, not math.")

## 2. Profiling with nvprof

On Colab or Linux with CUDA:

```bash
# Basic profiling
nvprof ./my_program

# Show memory operations
nvprof --print-gpu-trace ./my_program

# Key metrics
nvprof --metrics gld_efficiency,gst_efficiency,shared_efficiency ./my_program
# gld_efficiency: global load efficiency (100% = fully coalesced)
# gst_efficiency: global store efficiency
# shared_efficiency: shared memory bank conflict rate
```

On newer GPUs (Ampere+), use **Nsight Compute** (`ncu`):
```bash
ncu --set full ./my_program
```

In [ ]:
# Quick profiling on Colab (nvprof may require sudo on newer CUDA)
# Use CUDA events as a lightweight profiling alternative
%%writefile profile_demo.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N (1 << 22)
#define REPS 100

__global__ void kernel_coalesced(float* A, float* B, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) B[i] = A[i] * 1.5f + 0.1f;
}

__global__ void kernel_strided(float* A, float* B, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i * 32 < n) B[i] = A[i * 32] * 1.5f + 0.1f;
}

float bench(void* kfn, float* dA, float* dB, int blocks, int threads, const char* name) {
    typedef void (*KFn)(float*, float*, int);
    KFn fn = (KFn)kfn;
    cudaEvent_t t0, t1;
    cudaEventCreate(&t0); cudaEventCreate(&t1);
    fn<<<blocks, threads>>>(dA, dB, N); // warm up
    cudaDeviceSynchronize();
    cudaEventRecord(t0);
    for (int r = 0; r < REPS; r++) fn<<<blocks, threads>>>(dA, dB, N);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float ms; cudaEventElapsedTime(&ms, t0, t1);
    ms /= REPS;
    float bw = 2.0f * N * sizeof(float) / (ms * 1e-3f) / 1e9f;
    printf("%-20s  %.3f ms  BW=%.1f GB/s\n", name, ms, bw);
    cudaEventDestroy(t0); cudaEventDestroy(t1);
    return ms;
}

int main() {
    float *dA, *dB;
    cudaMalloc(&dA, N * sizeof(float));
    cudaMalloc(&dB, N * sizeof(float));
    int thr = 256, blk = (N + thr - 1) / thr;
    printf("%-20s  %-12s  %-12s\n", "Kernel", "Time", "Bandwidth");
    printf("%-20s  %-12s  %-12s\n", "------", "----", "---------");
    bench((void*)kernel_coalesced, dA, dB, blk,   thr, "Coalesced");
    bench((void*)kernel_strided,   dA, dB, blk/32, thr, "Strided-32");
    cudaFree(dA); cudaFree(dB);
}

In [ ]:
!nvcc -O2 -o profile_demo profile_demo.cu && ./profile_demo

## 3. Occupancy: How Full is Your GPU?

**Occupancy** = (active warps) / (maximum warps per SM)

Higher occupancy helps hide memory latency — while some warps wait for memory, others run.

Occupancy is limited by:
1. **Block size** — too small → too few warps
2. **Register usage** — too many registers → fewer concurrent threads fit
3. **Shared memory** — too much per block → fewer blocks per SM

```bash
# Check register and shared memory usage of your kernel:
nvcc --ptxas-options=-v my_kernel.cu
# Output: registers=N, smem=N bytes
```

**Rule of thumb:** Aim for >50% occupancy. Perfect 100% is not always necessary.

## 4. Optimization Checklist

Apply these checks in order for any new kernel:

```
□ 1. Is access to global memory coalesced?
      → Check: thread i accesses element i (stride=1)
      → Fix: reorder data layout (SoA) or use shared memory

□ 2. Is data reused multiple times? If yes, use shared memory.
      → Compute arithmetic intensity
      → If AI < ridge point → memory-bound → tiling helps

□ 3. Are simulation parameters the same for all threads?
      → Use __constant__ memory

□ 4. Are there bank conflicts?
      → Check with nvprof shared_efficiency metric
      → Fix: add padding to shared arrays

□ 5. Is occupancy adequate (>50%)?
      → Check with nvcc --ptxas-options=-v
      → Fix: reduce registers, reduce shared memory, adjust block size

□ 6. Is block size a multiple of 32 (warp size)?
      → Use 128, 256, or 512

□ 7. Are there redundant CPU↔GPU transfers?
      → Keep simulation data on GPU; only transfer for analysis
```

**Proceed to:** [Exercise 02](exercises/ex02_stub.ipynb) — optimize a naive matrix multiply step by step.